# 01 — SDNET2018 baseline (YOLOv8n-cls) on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanbahcecii/surface-anomaly-detector/blob/main/notebooks/01_train_baseline_colab.ipynb)

**Runtime → Change runtime type → T4 GPU** before running.

Before the first run, add two Colab secrets (🔑 icon in the left sidebar) with notebook access enabled:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token)

Data lives on the fast local disk (`/content`); weights and metrics are copied to Google Drive so they survive the session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/hasanbahcecii/surface-anomaly-detector.git"
DRIVE_OUT = "/content/drive/MyDrive/surface-anomaly-detector"

import os
if not os.path.exists("/content/surface-anomaly-detector"):
    !git clone -q {REPO_URL} /content/surface-anomaly-detector
%cd /content/surface-anomaly-detector
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT, exist_ok=True)

## 2. Download SDNET2018 from Kaggle

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

KAGGLE_DATASET = "aniruddhsharma/structural-defects-network-concrete-crack-images"
if not os.path.exists("data/raw"):
    !kaggle datasets download -d {KAGGLE_DATASET} -p data/ --unzip -q
    # Kaggle unzips into a dataset-named folder; normalize to data/raw
    !find data -maxdepth 1 -mindepth 1 -type d ! -name raw ! -name processed -exec mv {} data/raw \;
!find data/raw -maxdepth 3 -type d | head -30

## 3. Train / val / test split (70 / 15 / 15, stratified by surface × label)

In [ ]:
!python scripts/prepare_data.py --src data/raw --out data/processed --seed 42
!cp data/processed/split_manifest.csv {DRIVE_OUT}/

## 4. Fine-tune YOLOv8n-cls

SDNET2018 is image-level labeled (no bounding boxes), so the baseline is a **patch classifier**.
In v0.3 it is run as a sliding window over large surfaces to produce a crack heatmap.

In [5]:
ffrom ultralytics import YOLO

RUN_DIR = f"{DRIVE_OUT}/runs"
model = YOLO("yolov8n-cls.pt")
results = model.train(
    data="data/processed",
    epochs=30,
    imgsz=224,
    batch=128,
    patience=8,
    seed=42,
    project=RUN_DIR,
    name="baseline",
    exist_ok=True,
)
SAVE_DIR = str(model.trainer.save_dir)
print("saved to", SAVE_DIR)

Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto

KeyboardInterrupt: 

## 5. Evaluate on the held-out test split

In [ ]:
!python scripts/evaluate.py --weights {SAVE_DIR}/weights/best.pt --data data/processed --split test --out {SAVE_DIR}/test_metrics.json

## 6. Save to Drive

In [ ]:
!ls -la {SAVE_DIR}/weights